# Privacy-Preserving Data Transformation

Protecting personal features with an invertible linear transformation while preserving regression quality.

**Result:** R² remained 0.435 before and after transformation.

**Methods:** linear algebra, privacy transformation, Linear Regression, invariance proof.

> This portfolio version removes course-review correspondence and repetitive instructional text. The analysis, models, and reported metrics are based on the original completed project. The source datasets are not included in this repository.


## 1. Setup and data preparation

The insurance dataset is checked for missing values and duplicates before separating features from the target.


In [1]:
import pandas as pd
import numpy as np
import warnings

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

from numpy import linalg

from IPython.display import display

pd.set_option("display.precision", 3)
# np.set_printoptions(precision=3)

warnings.filterwarnings('ignore')


In [2]:
try:
    data = pd.read_csv(
        '/Users/kolotukhin.md/Downloads/jupyter_notebook/8/insurance.csv')
except:
    data = pd.read_csv('https://code.s3.yandex.net/datasets/insurance.csv')

#pd.set_option('display.max_columns', None)
#pd.set_option('display.max_rows', None)

data.info()

type(data)


pandas.core.frame.DataFrame

In [3]:
data.head()


In [4]:
data['salary'].sort_values(ascending=True).head()


In [5]:

data = data.astype({'age': 'int64', 'salary': 'int64'})


In [6]:
data.info()


In [7]:
# display(data)


In [8]:
display(data.isnull().sum())


In [9]:

# pd.DataFrame(data.isna().mean()
#    .style.background_gradient('coolwarm')\


In [10]:
print(data.duplicated().value_counts())
print(data.duplicated().sum())


False    4847
True      153
dtype: int64
153


In [11]:
# data.drop_duplicates(inplace=True)
# data.reset_index(drop=True)
# data.info()


## 2. Why the transformation preserves predictions

For an invertible matrix P, replacing X with XP transforms the optimal linear-regression coefficients to P⁻¹w. Therefore XP(P⁻¹w) = Xw, so predictions and R² are unchanged.


## 3. Transformation algorithm

A random invertible matrix is generated and used to transform the feature matrix. The inverse is retained only for authorised recovery.


## 4. Empirical verification

Linear Regression is fitted before and after transformation using the same split, then the R² values are compared.


In [12]:
features = data.drop(['insurance_benefits'], axis=1)
target = data['insurance_benefits']

random_matrix = np.random.normal(size=(4, 4))

inv_random_matrix = linalg.inv(random_matrix)

PX = np.dot(features, random_matrix)
PXP_mines_one = np.dot(PX, inv_random_matrix)


In [13]:
print('Feature matrix')
display(pd.DataFrame(features))
print('------------------------')
print('Feature matrix transformed')
display(pd.DataFrame(PX))
print('------------------------')
print('Feature matrix recovered')
display(pd.DataFrame(PXP_mines_one))


,0,1,2,3
0,-53379.658,30423.621,-82309.659,-102070.607
1,-40894.132,23272.903,-63048.769,-78187.777
2,-22598.537,12853.280,-34839.305,-43206.442
3,-44879.443,25611.934,-69213.084,-85824.046
4,-28088.113,15992.757,-43305.759,-53705.409
...,...,...,...,...
4995,-38421.345,21900.878,-59246.226,-73467.051
4996,-56393.134,32167.288,-86965.217,-107840.705
4997,-36484.823,20813.126,-56264.607,-69768.078
4998,-35194.748,20065.199,-54270.301,-67295.181


,0,1,2,3
0,1.000e+00,41.0,49600.0,1.0
1,5.457e-12,46.0,38000.0,1.0
2,3.638e-12,29.0,21000.0,0.0
3,9.095e-13,21.0,41700.0,2.0
4,1.000e+00,28.0,26100.0,0.0
...,...,...,...,...
4995,6.366e-12,28.0,35700.0,2.0
4996,8.185e-12,34.0,52400.0,1.0
4997,-9.095e-13,20.0,33900.0,2.0
4998,1.000e+00,22.0,32700.0,3.0


In [14]:

features_train, features_test, target_train, target_test = train_test_split(
    features, target, test_size=0.25, random_state=12345)


features_train_cod, features_test_cod, target_train_cod, target_test_cod = train_test_split(
    PX, target, test_size=0.25, random_state=12345)

print(features_train.shape)
print(features_test.shape)
print(target_train.shape)
print(target_test.shape)
print(features_train_cod.shape)
print(features_test_cod.shape)
print(target_train_cod.shape)
print(target_test_cod.shape)


(3750, 4)
(1250, 4)
(3750,)
(1250,)
(3750, 4)
(1250, 4)
(3750,)
(1250,)


In [15]:

model = LinearRegression()

model.fit(features_train, target_train)
predistions = model.predict(features_test)
print('R2 on original features:', round(
    r2_score(target_test, predistions), 3))


model.fit(features_train_cod, target_train_cod)
predistions = model.predict(features_test_cod)
print('R2 on transformed features:', round(
    r2_score(target_test_cod, predistions), 3))


## Conclusion

The experiment matches the algebraic result: the invertible transformation changed the representation of personal features without changing Linear Regression quality. This is an educational demonstration, not a replacement for production cryptography or formal privacy controls.
